# Pika H3 Flex 官方网页工作台
选择 A100 80GB / 高内存运行时。若已有测试正在生成，请先等它结束。
Colab Secrets 添加并授权 `CF_TUNNEL_TOKEN`；可选 `HF_TOKEN`。工作台无需密码。
运行下一格，通过你的独立工作台网页连接后端。默认无需密码；已有环境与模型缓存会复用。
Cloudflare Tunnel 服务地址应为 `http://127.0.0.1:8188`。首次模型下载发生在网页提交任务后。

In [ ]:
# -*- coding: utf-8 -*-
"""A100 官方 Pika 工作台一键启动：复用模型缓存，启动网页后端，无需密码。"""
from __future__ import annotations
import base64
import gzip
import json
import os
from pathlib import Path
import secrets
import shutil
import socket
import subprocess
import sys
import time
import urllib.request

PUBLIC_URL = "https://mcp-mac.id96.vip"
STUDIO_URL = "http://127.0.0.1:3091"  # 电脑上的本地独立网页；部署后可换为项目域名。
APP_VERSION = "1.3.0-87fc583-dirty"
PORT = 8188  # 复用原来 ComfyUI 的隧道服务端口。
STORAGE_ROOT = ""  # 空白自动检测 scratch；也可填 /mnt/local-scratch。
HTTP_DOWNLOAD = False  # Xet 下载异常时设 True，后续生成进程走普通 HTTP。
BUNDLE_GZ_B64 = "H4sIAAAAAAACCr19aXcbVbboX6nrviyVEkmW7DiDQawOiUPyCMQvNjT9bD+tslSyC0sqUVVK4nZ7LTMkccjI64wkNEkTIE13BmhuMBnIf+l2yfIn/sLbwzk1qTQkzWpY4FLVmc8+e977LA0YNdvRKhXdKliNmmNU9Ux9cWBUGfiN8rpe0y3N0UtK2TKrijOvK0d0yygb8Abq1EpaxazpitfAy4qtO426YtYqiymlZipawzGrmmMUlTluyjBrmena9AD+694+664+dFcfKHvMijartJ5daN0688uTz9x7V5uXf3LPfbFx8Qtlz8EDvzw5s3H67xt/O71x7r77l4/gp12Etorzinv3qvvhnX+tfMgtTtdonIVCueE0LL1QUIxq3bQcRavVTId6t6dr4t28Zs9XjFnv93u2WfN+mLZoq645WEo2NA4/vUL2fMMxKv7PxmzdMou67fdhL/rPuLLTtT2H9o4VDo+9o+SVxLbZ8k5dG8nu3Lkrp20bHsplt+s7suXtO0ZKQzt3FId2lLNDw9ntpcR07U2odlDWy5ZK5ZERrbQju31HUcvpO7bvzA0NjQzPlnbMbtuezZbKO3eUd4zsSmB3b+078DpUWkrYjmlpc3rBMk0nMaokEikl4Wj2Aj47Q0c0/G3ptllp4Drh2207s3V8W2rwzsG7Efip2XW9SE3kto/uwgK2rpfg97YheC7WGwWzXK6YGr6atBo6vIR1qdadQtmo6LLrsmHZTsGowpDkq4oWfWPpZd3Sa7Cm8GZqBt7MO069UDKP1kQH+7SKrS9P115/DSaZyypbtii7AMBqJb2sADirmjVnp5QtKaV4tJR/C6A1pei1I+KpYs7RU3J0uqbAP1gYmqkYtqNWtbpqO1aKXiaTXKBuGTVHTfw3jk1JZN4zDe4hmVLKlYY9n8f5irJwJmDX87TvmaoJAGjWjKIqvhpl7F0xbAUHIPqnah4UZfzx4+DhPx47/Adv5vXiQrA7HdYh0MxRw5lXzLpeU6Eb3LQEVi6aJaM2l080nHJ6ZyKpaDYcg1qpEqzJ8zSLMPTAWMaprU6jsZ2S2XDywfIHxsfovW5ZwfcTk3sPvT2ZUhz9mJNn6IgOC95YlmnZedj+ekUr6vBmtlG2jT/o+VwyPFDHWowMHf8pmxZsImEmmkqGxxdT0t9ULI9jKeUR8tp2M/oPL1vmqGU4OtXtXo7aU2PKWLjONMajmuHElQBIsYodhm5phq0rhxltj+GqqeWE++nT9ce33dvftX74StGPGU5+ySou//LkunvvTPPSD80rX7nPrgCaXQLIWE5EetSPFfW6o7ym2foYPcK5H40dFA26blYqajIGjKPQlHF0q2rUgJioHVYqfitDbdAS4VQR2HIjHRoSUwgA3SRXGTtWNyy91KuXBQPn1M9Q4vYTdyT8ugzTrlTiJheAzUyxYtre2jBIlhObVx827/1Pa+VK88pD3LEoLlHSjGZGM7nysh2FW8aCiCgLRbNWNuZUiej4J0BeySg6KhMJ2bUGCBfxFpI6tUA4u1BIZog0HIERZkQJAIG5ijkL+BfezemAFmVhQCyIjaiFDCAKOatiGbsU1QeBjuUKTOqBzjdXL2weP7vx9F4G6XDCQ5JQJ2PY1LA3ekaT9XoF+ZA8Ee4MztJWsbSlw3wRvagRxJK2jblEMrBn0DysJAAvMTBAZVTZaooWJqkAHgGOxntN6w2/efmSkR3lw/iOVmmIo5jgGbnPjm/eety8et+9/1Pru1vAvqyvXXTPf9v66On602cbF+8ogkqvr93d+Oxj9+4V2PPgueTuMo16CY+PNxi/gAcu1F/z86+ajy4guMBywLlfbX19YuP65dbHnzVv/H3z1sPNj2SPvzw5FYfr9GNI4Gltl5g1GFVUIrQpprdA7hKGeM843HtdCRenVpHaV0LF5Wug73HNL/vbT1OfYkZlhrer5g0wsAHti49VYEnPNi/eb575IJFsazPA7HgtL3lMDzBV9cRy1x78BhTYTu5HwfoA3FTd79NZrOuq7Njjp2YIc3LXDgIbPo4or+SV9qL4NjfSdTyysOJBnDLyz5U/5UYUACvE/JcexCyDYOcCSzCUY7ZOsnfDo0P4Z9voMP4ZGd1GH0dz/HKEi9DLIS6yazS3nf8O5XqsIfceu0+hNSMWM3a9ssH14mLKK8oQcoLbh7v2jYWV1pmP3Os/YP93v9y4dU/2jyzEgr6IC6KGeNo2HjSIBiKjhgb8Ic+aZqUXxignlqDOsr+B7tpH7oOL7sqTwLqo2JK3MEE+egbOVOhjgKOeSSaV/8p7R2cqfLJmui7U5teX3Qc/u2vfNC+fXH/8kEe3vnZOoTMGaM19dDF24YJ8f4TnD/P7kUUMrB/IbiVFxQUkghRcWZ8wxKPiffDtLdPZZzZqpdD6jipLgXY8HgiwEbL/Pn7whI8Zb/EjBAOrpEhgIGKhRvBVPu/hOFp92hysk+y63uuPjytcTULC+QvrP19X/BEp7uqV1q07wUWHj7jo2Hqg8QWjBtTM1uspEmVhelAASbhjIKpQE6OJdooIxXE2WNfHCt62HTFKuokPWqNkmIllmri3QdhLz50Jgbx7/sPWykcMWs0rN5uXVpF2wTADGwPifE1sjORr5uBQ1i2dBCGPNTBqZTMsuND3AnBY9YajTiVqMHpDS9tVA2eQTr/f0K3FNLSVh8nqx1I1raqnqnrVtBYzjuloFfmjYQNbULKMI7pVgP9sRMrUAqx9VXPyRftIqmbOA/OhW/DQqBkOwE1A0AmLkK+Pv61wjwp1qVBnypvGaykFu+In7m90erqWULbS5GLINZ0qmDR+Bm7SMurAkNnAIzgolwB3NpWd4d9qIhXAI4jPgGvA2lNDgKdeUXaMZLPZNsgMiRcJ9+mf3FNnFRw/Is2r9zf/8nHzxhdAYZSdWZDCm7e+BA6ndf9HQKate7dZqQPsa4eOh6HjV5Vt2V3be/SLHbo/fu+evQkcU+vh8dazk8q2143X/rXyIXTmHl91P7jRvPsXZY9ZLS++fQBIsHv84frjy81zdzYunHB/erh58pPmxZ8RnnU4hgtIE2kWzTu33D+fhjG7J85unjzb+vFTEJWAU2J+CebYOvmt+8md5ucrrWfXN+6chg7lXAA0JA9NMqBc5VGAjFwyJIbS2UgMIlQOQjXcq0QywKuGd0xyqdoRzahosxWdttdRoeZU4k29ulu+T8idxU1OKluUXHZoW1iGcM/fhxXb+MfjjcdfuCeOu3ev4vHyWx5UXn+N5Af4G8cNwob5hV9RcqhkwSo9wIR65f5wDR/+A2u+/hrDBmwXb4fYI7EF11vPriHRu3AfFty9dmf92efuvdX1p9c2T57HDYJN/PzPLDI0b6ysPz3X+vqD5ne0HYwTbL0C1K0gVF0RPj3A84R0YUHaV5wHKawmpZ/44skMEFGgSnBMLTUgGLWhUW4McWHJsNR4TBhetWBHeLxQv3njjlizW39zT1xz7/1589rx5pkPW0+ftn687/78cSJeAQSyghYWjjojRZBQS9Waw/iM5C964u/4PLn78Otjk6l9E5O/Hx8LY7WgnAKLYqCMglR0agY3xRsNbM4RoJuOajh61Y6uBZ4U/IAnhQrECMsg54KUKbcGSyGZxXewJTE6IBwElmLZtGwjY4ZzScTrWMq2T+qcar1sk+ZTP+I949qRsnPR5hf2+w3NnofnZeJREmUAiIRspGzDqh/V27e9P3VHwwYwQEJGKuZMybAXCvRO5Sl3UU/AXKgkky/l1bwynOUjS8M07YxWRCAQLaXwze8Kh95IdhlOeHczWr2u10qqyv2ULR3pF4+ru07m0AQBejcNjIbq8+hbH3R4N4vzRqUEnBBsw9RMsE8uiLCPUF3RYbMQnEIQIk+nN5/owWzU4EyUynQI5ifboCvu4DZvfNv8cqX5P6fd1QeA+uCgInl5NQ9LjzTxxt/dGw82rl9lYrX++JxSKiutn//knnzEZFKRFg0+04Dp+LgrQYwACBJwwvqTzwRFuvgToNj1R59ywyGNgURjVe2Y6k80hZx5vqJVZ0sarLRhjdL/kXRM5QSP2xnyuFGfKgCrId9lSAtlo9IZiBygTUevOYPEvSSSPSgFLMjmyqnm6b/yKilh246c1VHTWkDenCc2qCTGjQWtsH+4sK+iHyscKpeNoqFVhN6oqAGCQ/VF1heaqAGoN19OJONQMqKgsllB7k0pABlE4MGDDKfjqFZZCNWPwV7IxNG5x3pxqj5mvgl3cTdJaA1rxWIjBFCsgiO1F6vAFiy8ICYRa7E1z+3BRhGzAUTS+IP+qx1WS3+/gbpVAXJDI4RwUsrQiEA9aTGQiIKTN5l0VbSzy8oflY2/Ptq88r2y5GOXEI8CJfbvUzae/AnqghzH0+uDiQk094o33u6wWU7QWH5gBgY5EGJ+gePduHeFeY+NR18rS97sQ+PkeQQBOFNdQLDTj4G8WDBD5hsh4mApn5kpwlsVYURu/RGUm2CNASh1kGQsoO1zoghRNsn+h8RVqhSYaDu8kKVzzjTnKnqmSLZYYbNEFgeRabi4HIUqP3uDICG4bRwBmOpkUwhCEi0E9SFXQq/ZaM49AnNWAUycQn0xpRwhyxP9r2J6TF59EQaGn/CszhogN8Hf+qIzH1Qq1xcztAd26EzVjXqBGKOwAEnkAHFdfRFVlukqMQNGnXkkKQrOBG1zWp3Mz8w/CStXVA6UQ5GdZnjiRbOko9ogGyVKvCz1RcHs0YpA5Q5jFavkD9ir8G8O2yj7fQeH/F/hIUMxgFEAC71hlFTSfWSlwk5QlqPzRhGoBXSaRh6uH/Z448kj98GnyjhtpzKcyQ3xVrPqULDJ9061vjyu8JZjkTQVcS+cY+Eu0aY3T7RufbX59EK0CnodfPwQ6rmr193HjwCO1p/dc2+fbV37rHn6C3ZF6GQpZBZCTg1WnHX3tPbvvx9adbRBI/x2qSz8K6j2omwjJcHaG3CXZqlJ7+QIkKA6KSLj+JjsUT1QsysYRar6cMtnebYBrFuhaFarwJaoxYqRgjJ1MyVkM2BHCfy880waxIBWLqhSnCEDDYwfW8CDrh/TqnUgwHzqqSQ9q0ILl3GOOYkYu0JAT8emKzJ7UGnJXRarSNzocMGg6WClqXYq0hh9qcKZqKRxWLhQyKukD2qz9iA97R9OI9fCS2jpyLOSyOU5edAHHn5a6E75l6jh2S1SscYMKuVZDniHY2wPVExo/1Nt5gD6Sur0cANCER8WELGA2LeZNmE7qD0PydrVkpQkoC0olZalwqpkPGPaHCmU1agKOZ2mF2mhRU7BTge0ylQAf8vvndXMEeyDYwPMh2Obwt5TwbIz7VrXOJ1xZKqyOdpBFC7g/zMRBWe1JI9JHWBHA2QsyHwV+Gk1oMgAuSZTr2gOKh8RuSaAQ2wcS/RmtaNOVwexHunB3LOX3OMfNW8+YZVdIujGQoxdvCNLF1NyWEMb4uPjFTX+skp2GrbTnCNpm8GLHkEkF1goTewf2WTCbLlk14kp6cJ2IWkIsVNFs76o+h+l0XX/vsL+Q2+O5RHQg5JASsEvb79W2LN7z/62z/SnMSuKvTs22bnYMSSBXmt7D0zsfu3gGFbJJ3IhjBU2Ps0IdJWFNRg/MM7tQ/XDoT78lYIuJt8cj37HBYUv47+f3H/orbffeu3tffvGDo/txa5Typ639+4uvHNg4gAOaO/YOwf2jE3kqUP6Akjrbfhw8NDuvQfeej2fOLj7//w+QW0dOrxnf4HK7D548NCeAsJGPsG6M1QmFmx9DuHaHsUNkfDmmAskuQruNwErMnnojbG3goZBLBLYbNinKb/cDIIqlvDgt1qX0AuTLuODmnjp9y9VXyqlX9r/0psvTSSSylZAFahjx2Whovi/Qg04ROUlJZelf8QI0G8rr8i1I/iEh3ICqGR6ibpbzsDbRETS+fGr9ac3Nq7fc59eQnkH6y9PT9ci7jgxDEXVsG2jhn1O0bEIn485g3iEOsrMZGMrV+v6HD7NFUlfVdUWYHE9006Q+aLT4SNs0VEPNq5vPk2onH/8yj3+I8wPVxfHw15zoq/kf5xp6rxexJmkgY0GmERdgs9hpUt6N+5KMFYItICW9WN6seGQtlwgasElFVDpPzU6ROyGOpxSckPCUSbMEftsXSIsyIl+emneST/O0mmQVfbEV9RVnfoZSAAq3D3lC1sYg6w1arCCnrkBIw7xW/4ZsM2GVdQTocFikTgpi94zQm7bPrE3Rs2QTAWWTvaz86Jqek+wInmYVE2GIa1ExnvTMuaMmjTj26ODg1AVkDRg/upgPKOWwca7bL+ulWLEsC5DOpIG2m7TqPaP7d6LbfcQvqSXWFS+p77hVEq34qABwLCcxa620E4DRB1Rw2buqW7CzlY0oxZnyxSDoJ7k0PpAD81HFzZufsCosHnjFOv9mhd/cj8Rpo71Z59vXLoGolfz1Gm0BN3+eOPCCV/7Se5Mib5BoKyLo47MYsXRrfxsxZwdrZk1wSGW9LpDBXIhAJFr+u+AH6242XBkR45GQ+nd9BHmTOQJw5/p+uIw4ISAnqMfvYjHykwlxndP7ifyKEU+qSNB8gd4HhWEaPrfSpwPKdqpSsBoUgacrVtE06Ad4UKfsee1oZHtquo5tzMdrS9iwx3hL6BTeSdiU8qQ656uYn1PuBNaNmIZWDRj++nsokM208y8fqxkzOm255hZ1awF3QosY4CbTmPdxaC3Ib2AwuSD5hNFaiPWB7FdkSabCFjdRP2ApTdJBje5jCx9Bl606cxU31sipbyhL9JTsqMCDf2FcRghq+ZzabWo+ItptIisdu3JMc2KPQhyX2WxUDLKaDSz7ALV0wEEmVSnvSGgrznrB55/MHI3YvVrZAyjbqNfImQsvJQdVCFBbiNdMmwk/2lkuwXtFxOiz436nKWV9EBtipqhhcFfR+d1vdIPu/NvDcNKsOIl9mTFL3y3YfTc1udvsvPMaibibKZQupjHC61XO6SHqwIMz6Kme3pABvEAOZx/WSoOJ/HXaCJFbzMF6RdUKKQSKPd4X8T7TLFR0pIvK3BUddkWvUPs4vlYqEnZfqAAIOMCMKFGEWQSYNrVbBJKHctzCRN9RtThodTwUDLFpfIJrJZIldDqLYrNkt9NbrvXvnrst8eAs2hU1WQGbasqNhro1F6sFecBXRp/QO+GgTZdYJH1UrN65wUsVow4fTwyV/PDqOJKBFolZSBs77xeqfcFLwK3UhAGI1dCvKVGtW6rSyEsOxpEsbjzcNoLnu5t1KPIy8mYOJmQkoakZkFgURM4XRtIKQMwlTkL0B3G0NVgTBxCx2Fph2Zhv4/oFD9nCkslKv51cg5CoymgMkUvGQ6KeUd1Y27esYGPQMtWA82Q5B1hK8xmZ2SsmwDJcqNWpNMXH8nGYWfTtfHDY/sOvEuwPDV+4I3dhfHDh14/PDYxMaNgY56ppWo4ah0IO+DUKsxHm4OHLVvIHGP7SlkEH9HiViWw6KQEour5cCP59sZSkn/R7KJh5KULeSSEQdjCHFQT0VRhvwDsQo3Lcf3WWwuACQ2GIyuIHcSW8ENdL6lbRGTYloWjFMEV1Ce0r0GIqhAYyKbbGgrb87i3wERQQQ3bXDAZKKzApALn7UVmJEIskUvoMjn2CskrFb2m8tep6QHbmKtqduHI9MAMxjjkgrEHDWL1soHQix4aQW8JoV0x22k4ItMD7pMV95vTLJ9unjzZvPkjvge5q17RocF8VnhB5un/yaj/kFewYDt6Xa2aeDhSilFDvWDUbiD/AS6/YhZhxjSR8LffKId1OiOiZXSd18ogIJA+lHhGQBqGPY+eE8gMVHVLrywqe+ArEFdmHsJNhlFmWKVMK7k1H1xd9tuCfY1dThndg/ATqtG+uOXpAV5R9GyGfpYHl2gRl8MrTN9Cq5yKse57MGrrRbNWsvMWelCrYqApBahMt1og5MTXBNTPi7BFUanzNP1MorNkaLc5aA5WxANp4NHmDNh2q1A2LYDaUmHeNBfUMEgku7Pm4WPb5bQEnEJiQ7dETB+qFny/v04Hvgqysyc1yJ3DXfWPxerneCzOXBaejeSwunHntPvo/PSAFPUD7EfoDZLSwvxwxiMnmu2RFiGyURizKFY0LV1WxR0iJ3AYSYrCxND3RUfsrDFCOqKJNmSL0tmxgCyMY5MoiYi5w3ecHT/xTNFF6ouv0JK6drr19KmYLzl/bjz+eP3xQ2++wbFl9g/vkT/HiE23hHzo9d9fcRxCsCQPauPJ5Y2bHzTPX9i4/ah1/+z62jl2SN/44Wbz8wveiHh9Mmy+dCytZqPthaRLHkOnAtgHfvP3+5ObMPn1tcc8c+hwc+XDzeNn3Wt3vN78XcjQudIL/ptCRSuVuOM2YtJHvVQHFk8qHjQdVnCvTgtnA1OL2rhCwZtmh+84sZLOfFNwmq1vvoTVRV/etXP8vPnFPzYuPvZmiu0xA1etbwv14r3F1iRPJtq+sNpcvaC8Ob5NcX96yGAVaTcM9sCGCqjH84iHRDJiBXzBVUKvPCMQH9kSMNncd+ubE5t/+RRdk9Gz+Vzz5q3Nb89gaO0qAwv+C8JjgRh1XDhkuQrUZqEwPTAqWVfuAnlH3rsQy4jqR59X3HPwgPJ+Q2/oJLTCxupp1lIpVXPWAExpOxiKkZHqW87NYJWUijFradYiJWTI/Ao5EuarWjGezawC7UJZw36+FAqW7udKIIOP3TG7gjEHuLhbsgVnHoV1hJEw70uDQGWvj/jMBUO3A18yvAWyAEY/75+cHD8McrFuO/sJ38PBmZQ94McJqiIaaVgVVISRWtdDzfij8D5gQfhKPxA0gpkcpgf6TeWA+7Z7fLzwztjhiQOH3qK6ucxwJpuG78WRncNpUsJiscndE29MYOjm9AA6MQDEtQVvgjgrv4TDNwFTRapwpGaA/k0PlCvhyiKYc3qAPSjaO1yGsU+Mj+2ZxGEBt4lxhXyWMFCQn3KjOX7A4EN+Gh4d4odto8P8MDK6Tb4ZkWXEmyFZBmMOgY2dro29Ozn2Fi6WWAxyAcDBwXOmLjFJ5r26/6TLx6P6bH16YLmNzZkeoOAn2QrgJlEBOAG/ahWrIunDQynLHtVkiWpdDDVTrsBxEi+3wcItw0od2EsxWmiOqKPC0Zoe+L9T2fQuLV2eWRoeWv5vQjJv7n638PY4Gl2h9BD5d2Pkx5YtQ4yAihWgu8oEoYVRX1SQ2Fq19Uo5paBTMQmxKMeDLKwfFVJYMNAaSma4ID1SaXoSVaRXK5ZBIzIFTpCHjCgQ52NLtsvpgfeAcPECCPM+/2jUSXcKM41wXqo3GrS4shutMCZxWLktc0rE2Pq92aCuFLj7Elv+0bpZOwKDIal4YvLtvQcOFcZ3T0z87tDhvTwej6h4TdiAeJCkYqSjtFXbGTI7C3308FC0DsghqIT10FTm8EHTd4zwih3VFvRQsbEj5PwRHYFjIqPZR0lcZDwCy5H3GjDCR7ArTN0QXSJOABLzpawZFaAb0RZ/ozSv3Nz45GFz5YPmqW9at84Ayf/lyWfrjx+7n9zC6OgfHrs3RDzQ5skz/wKO5+RZDPS5cG7z6hfu6pXm2vGNvz5CI+WP33M+ALQ/IoVff3wWqgr5zh7EbD/hIfEChFaCMbUIccjzupqoLEopJQ349losYHB9dmZXPYmExXbg9PnQwHIGwZKpqjd5996fN849gJm3nl2DeaLn8IWnG9evYtQ/WbVaDz7CcKwPf2pevb++9mj90QmFR4b21vXHyCFBJYydWn3g3v64efGnQHqjwIAYAGlMId0F5rhYFDYeUiJKZwY1idaUHYApdm7f5jkzBEQXUXUroqPpAfiL1D5T04+qUZBPibKemSYVMQVFbDH+sIEZNUBYCww+xc4a3e0q3F0KmQDp3SHi3misKEC2VZApDH61WcRIk7jA3AhGMQaWmhg16hnxODpyiXZsdHGUo0t2t/XsdhzLmG04eqzFRwxBWKv8JWbkiar3ebHApG/zew2lPIoPaxY1AIaS0mv3wN5MuVGpVFHpwd97hvaCbEUyVPPmydb9EzK0Fw28cPablx+sr32y/uRmCLmSCVKmYAph+yBVyGDiER4Eg+sWeBnTylSdSE6d8gDRO/S4ztgNYKyPodkcOQE468QvBFYDdWNUnjxecv3OMxKw13WSCsdj2FPZcNce4Arz1LwYrYwmk/FlPnczJavM9DvOje8es9zJ2BmHfeYnEEBDg5SOu8StRw4wpSEh0ELv/5APvXTX9cICpgf4XQwlbYc76eUaBLwc5negHaGP0s8gia93huOVO8wa9gJjrY5/peT+ufInrKO4d69s/P1rTIsh5f7QyCivQWgS+IanwEx112lgYX8S1JjYNmLO+9gnIoCoH3hy3kuNEdkeOzxA3wFYsMg7s/VQDS87SKiafIuVggmVRF6OUFl+J/h14uG7roKFalO5CDheGV8pB4ftYJ4UYHxpnTBzhhxPx8wo3jQoH4r8FOiWR+n3LGYiOheCSF9AI8PSTgHewsQ8qydaP9/ZOHfyX8DjXHm4eekZHKGNi4/dn45HGESiO4GFwzfRxaJSyAVzKjo8GiklnYsiVdGY5DAtQAuzOuACdWjLlhCLKVOPYIWOaVKoNfgLlUeGUdHfByh+A0jtAuX7eASNtE7+Q9mVze7I7do1NLJtx7bsrl05P73ML0+uA4uDgU+fnW/eeBSa828ooPmDz0h97p5daz19CmxSUJPmrq1t/vVx8+rPFKN+ZnPlC/fR1+tPrrrHV4EHQkhRYNFZ/9I6/RUX5Ngs98F3m9fOutf/TPl3QiyicCoPO3Z4eRNSCvqHh7cr4GPuYa1UsIDvYh6P1ThDCzUjM7JgFc7AQihgCpHCTF9Q2DxzqnlqBWFRoAOkJ15GllDXSOyYKiJgBeYXhSqj3BYVFmKEgzyE4AU8ATqEhMptWEj6v+PKTM30QBDh5CmMdfMBTUIkaQqWQkrAP15VckO9V9C9/ZlIMELLt3ljhahAbgjA8Vumhyh0eHkICOG2vv5g89rH7trXsapgz5pjRzQL2VRQQ5AN6QCyAWGpWNG1Gp3rqZnw7sUncem4fIE8Ze0fxZZgFhcCYB8VRz5J3MhzGu2Q1C6yrIGFYX6C+bvQIskENJx1Zkp2OBNFcRGI8wdo8MixWpw1zZ7CLzMxRjWxwjK+ZEl2PSoS4nDTo2JY+DyznIzEhlAH3u7OALztwuXzPoiNxg/DoQ9iz70PBK/E0FHceVvDW9vb7IO3bt5YAdiGMbnXfwb+eFhhxEiPm1/8Ax4Jrr9TGNTR2fck4MlP19dW3Cc3odLGqZPN1cvwE75y3TgGkC36xHrlwxxYijibPDM7PvuRJ7ovyXRePqTiw4iZOuc9og0EKo//SymBsKG8+JtSAqg5z/gtvlkfQ+fxMRVInZQXsBGW7xuzqOePZ2qJWgcUXjKOUajDYn0DO/i+wsEhH2wgpiIpwKUzm5//ZfMvH7e+WWV/bNR1sNu1yIMSSJHYvHEWGPaIMGFpVZtDasoZjz2nOYSI7+Nz7o/fAzOzeemGe/sH9Pd+dBsIcPN7TEHoXjjDgKDY85ho4Izo7/ID2OQFHTPiXH7gPn4kKp866383SmGNDNQvGFEGiF+GBi6cosIFxcsenKVorouAKkq0cVPe4MIqO5Dz1dz2GG5KjCfKUMmxv9LGQfmzykWS8Xo6wDbdBhBAju/w1HW0GnIObXTAq4EoRf6YIrOy07ARozCXTSajErMo6Jbkmc+KuIoV9h1Y7g/ft55dA+gQ23/3SwBShgAvn497/nLz4ap74ixDVhsdwEPUqKrvvfA4iT6+h1X8ZWJPIhQGXwVhsL+ZsLoRz8LjWyJT1d1T7s/HARfyHFgqbpsAdCij3AzAURIExYbnxV/AW5aOKdPzAW1QqmMKBV6KfHANyAAKb3hU6IqCb/mY5/lPMoaA4npMiVGhjzf8jlEsZzB9aTuSp8I+NmRVa1CFFK/0vH8eeahTq5QnC7EHJ3EDrCKVta+IEb2aPvKKWKFX0WwS0GlGhhLW+XhmAdT3Tw8swTimEkYpMbOcPrKE2jd4wZ7qMhSZ8EEuuczGmeCsKuZclynFds/2iefvG7oK9y3jnWO7rvN2wcng7Q1xSJ5G5dcZmXPMCS66oOgBD8r6lK+xmQn4Qk4PkDNk6Fjw7sixBaEGZxgKf5Wx0x7R9NIAQ3dktZY8qhpJPIzIk51ep2O8LFHxh6iCg7HppExJbc0Mf/BjsRmrxAf5wLe2U4q1pY8o1g0EaOOnQIQ2H1xLMEpiRGGVDI4r+GYmvr+gQoZanAoqaWZE074qBr/LX2LCrG7wq/NvWZW3SX4PR213iLqOEYlDgde0dTFycTD4GjeqXaSpxwVfxwdge9ATFBW4flA6Tc4kY0UqXn5fOu0W8c2bV8ZJhOQWUjaPkm0kdjQBaSJSMzyscLi3hyOI2vkoIsK/dGUimDj5hICrz7StN6MZnwCTtC1pT8w2ICKReaclqRJUOUytMDUjUU/yRmSvyRANjBpoKp1G43EAzzccwTTI8TCPwkkGOW8j+uqQxqiNsgdvPJDGz44p71kx33fue7YrernvsWKHFGkIAmWMtsOo+LraLR1b94xPHdLlD3XJGfdvpMzvMMkUz6XbFQqdrLX+IoSNtb5dtq1vMs+GaU48d/NbBBujWNWB4pT8oxc/9jZfiPZ15wBpisYm4K2btnEsFnShGF4sUJ9TGXhIfCGfpszEgdffOHDwIGUhxI5FULD3cXLs8Jtxx4cKd7pWIO4Wg0iKyN73Mwi4GGegOGiaC416XFIwjj4LoDK6zCBqmw76LUQZ0Y4cqhS+hZFfyGE+7BQbFjISBX6jtuvPfbs+BZ7L8zAckv/ZBN824qPz6F3nCfty7JiUTXfi+6I50MnLJTt9Rg2EFd2crvidFhlok5ATTXQKV9X3uslEhO47I/tkKBnge6NUVsguSLA6p6XjYXRACZj7z6g19FjcjUZPrh0ye8Z7pTBxMNr1hZ0IQUB0FG7zIRnMJ1UnT7i3Twq1Crkdex7Jzc8/2jx5NpZAdMMznBFC7N1cHAfcGWszb8xssRAS2qv2TFUSTUcoM7B382ras68w+fZbbwFHS4kz2r2tAj0DsaurxBWS80Ds6GC3ohlECOrQWhiUOfoG+G48S5Dqd6FOLP8fqAF2sxp1AAmpBRCOST7XkuxCvztdhwR79wJ3H3WnpfEEP+a2JE8g8+5LQgaXLkyidyJk7gV6o9NTqOlHpaNOXo0jcsnO1zLF3EAFUgoPq02e7HgLVdx1Tt5+dL/T6XmuaOrnmqagcwgl+vbzncbF0/XKpdubgRN0maTdcEA59j+FVrD4jmeSvRvu6+T1exK78eud/hHaDjE/xhLer+dshdB6uAl4K+L2YolHVza2A4p/PloXw1v3FcXff0Zkn/QcpVAQgkrhI9S5NK1Vm19nzE4DbGGkzAzIunDoqN1eMoBoundATRcxpEJdi2gZGQn7ImPg4BPlfx/Va8Ppdw6ibaz1zanW/butOys8PvfaHUnt+x4QhYvXTTQC2POaVXqOoaH3gwj4wXt5dqBJzl09sXHq5C9Pzryze0zxAmXw8zBb7FaFSW9I2SpMekrulyenyDxzB2o37591zzyCh5AFJh5UhbnaAiRn4MWdFfQsV6enS1uTg/wH0XMPNBkMIEU9o7BbpnMzU7mZpG82ZQkm270hyuRFbeWV4T4OQnR3RSDO2jleGm8Bu22ot6kiOXpbOqtuHZc9uBKKQ39P5VauirBL3r9TvYdi6y8w82CffYDvS38MwCkqgoGtdgbttpf2oOG80GFjVpoiFjlV1PraXYTe+1+3vvlAJjg71cdIKYLsRYYQDi3ritqoUo+Gn5tA/lrEsY2k8WP85Y8hhVXHGwlDXGzsbYP/aab8RZSKne4VqND1mH5Gl1hLREcLffiOgZJu9rZl9DMZGSzomfLIVCpjBPtjyWk4NGpK1Ug/gePm1xRikqfnSEb37iNtY7qW4AUmOkBfmGCQo8c+4evw8Jc77URXVBa3TBhKEtzwchiVbK6suCcfIU6l60tPsbG2+cVXGzdOM1LxZB+O8HCgPTWgNkhOpbejg/BM/B2nXi5yFEDg3egLngYPdD2lRZdjEd9EjxNBhNXrhWhr++qFjktMDYQimGVUEd/5dtK+Js8Tlgp/BqqA1qULOurBaAeUQam2mKRUBw62TYsYUPEicHSyrUTMJ2Q7FaVCmkh416b5JF/+brqfmJPBt0v4irw4266IchD2XRjKcvrIFrYsR/dQDIJaRWaMNMnUBe98vOVYzJAjKLjdNqcbeQdFvH8Tq9elq4JQqZDT0j9Xvg7COoEStUUKAaCLs0LwL0dXBnZOX1DxEolsKu7KCiWtDG3DZKzxuv5yhtXAGWIl0GbcRUuSDAZLiihfNT74V86dA4ULvpsP25M50nIwl8n6dv/fAsDWdYzK9YBQq9fbdMxBxwMRCQ7Foo4LAiELCC5XHZG2ItjSb5S9Ju0a6iNFkLPMHG6jH3LJ4GRCMgDSzrSFvVCb6FrL1BRwBqWIAe4+rPMYaVdy2I06XYgVHG5goMEZoTpDTMWcfS8l3WCGstmUGHhbkBKM3susx7mGqGZMCiEvnqstZrNWKgDpq5s1W+LZ2DJ8iR9Mcg9fZJOeXKwLuogJzowiGeMHcSgvg0yIkd1Ovt07okuDB/XanDMvbfC47ji9ZLJHbUzhnMY2LLPC46mZacyhrffq+N10cC7pQ3WO8xdt2DWjXI4ef96HOBNHuOkJ3YHGsSznmsGn6GD8Cna70Qdxi9DS0SoEAUVrOPOYjreIZom2o+NdV4mkol73Anvj8RTqTwMoX7PoVgE6JZkJA5Nl8DTUrnlkoB4p4hhni0kJ70V/GSgmIB5DeaMNx0JCu8idYcIK8QqIKRtzkqyRCX9DGQnnQBg+EtkgGBw5OR5WnN2uUwwh7kO7+Q6xL8v9sXOPgvb0QDa8CCh8YRuvAHbBixvw+VWRXLu/UIjvPmze/bJ5+WTr2Un39jfkvfxo46/hOJOYi++I/hGYEXEgPrmrgym2IXzr+x2YvBT3kfK/Jg69pfBd37EO1XSfjr/WJbPw+thkvLFUZpkKszJQY/zQxPNVkW+jddoBXHALjVhLcEiLSUuMs+VsMKzRjGemmbVCrA8SBxFiFC0kS7qcUsIRyaI3qSbtqzFp0/GiMCkLw7bstvZ2gy7ZLzjekexwzOq8ZqE78bhRl0sEp6KmU8KpwzpQiR6pW+OEkz7nzp7h7onjmx/dcZ986K6tBcUl4VDOhaTsRLPIhsCEtz0KIw0L9X0yiYkqcig485GQ1pRCl+ly0Qy/kTlQVHxFn4OVTNuJ2D5ZYELoDrKP6HqDZT0BP4x6JvRieh8meU5PABFBxpLFLsu07bRNr0Y7o+I4QFo72zz3lXv+SuvHOxt/uyY9mgmchiP3NxHrjd0NanVjEHgfA5EzOQnBkNuy0R3tJh/xdU7CQxKdGrRSiewZUWu5L5ohIekgnmmOoyPnh66XDhnRHM+ChtVkJgdaRaNO4VqMC4+mHUDS2+Ns9JJNFG2T43WukxK1xzK7Dz5v3b/U/Put5qUHhNE/E+m/11bc1ROb/+8LcWnT/Uu89kO74hy3/GQe1A+RrqQMPRYfY8L0ugQhi0rh8IK4YH5ZMpBBoI0X8b4lu3mIBfdjyqijFO1t31ZlCrZk5oWWuHXv2eaVe+79E5xWgu+4F5Dc7p9ShO5JpHlZmaAfrEiIOXLvpvdxHj69lB63TMf0Th1l1Yd6QksR1XjEjddcwMEie7bssf/lML+TX/KWSTJMyeWXKdlMfvBlZT90eqhWWYRxa1UdsUB+wrGAfL+svKkdS++e0/Pbs9t2ZrNLPMnlKLMbOcYw2QoyMZEtQ25UwlmYM40eZL6GfGmacnsZRaFIk3lvyxX9WJpzdjFkemEvo0ogyxPlxalRJj+xPhxRGOgYP+CLDjpFLl2Q9xNiaYqmbIPS5HKbXxvWjAFZumVc6Kpk0iC/OX6BYhngQUywq5UW/c+hYCkvIK6tdnf9KOuE/EaljoidoO28iPBO9oA7nEc7DASFXFUAA2/RYFUvGdpgvBuzP40wVMRFpPQ4sBRMw+kKgoc35szGUh8TXbs7kZ9+D1/k7D3XOcv2OFpC/SQ3P2aUfRE3kdRIKNC80iEPuO4ebuihjedO+KhOjYIgMtPffvEcRmkQy/3Otp/98JGcH48oKBrMZii0/VWRah8tul7cmxXod1AN5g1LDqqsEP4j5c8eYOVaePDURrtnOcU5aUWRPIIKZeaAVazbajLOwxyViiKkOLQxHeOzJJ/frnOVuEh07mvTOy9q94UVTvaio+49kU60G6R2gg+qOOpPXhomsMvu8OLluOkHZLiwGmWrOwIGY7AwVHQBhG4zDyjMvWOHnAFVRG1hHFxQg7EWEjY89VhdIhtel1ELnRd6P+hl5ZO3pLGyJLpMImsZiUReLjIvQVnz+1vNG6dYosTcXU9PcGxb6xmKlhtffrBx/er62ifwqXnmFIfAued/RPPLjZXm5Qcbn33c5pMRq/THqbRHuwndv1xNUtGj9p+mFoXaij6nFReD2xEbPhduTTQV8VAlB/wulgRV9EVSAD4FyDkViPEOFeSR2+4z2JQ8O9wHnzVvfBvIrNoVMORt9s8JBR3laZT/vozTJUQzbQk9NjccmB/7S/J7wTXTdd+YFwP54plkSKzTjzkykyJfWhjO++QXFAkVahispy6wty8m93L0GlJrO5wdii52EN7g2IVRCxRNtrnzis3CPnorvrzMRK37X/HpaN584j45H1ZG/wpaQ5G/RmgP84qf/bKP/B9nL3mJPjZvrLgPzq8/OocpM980XsODvXYR83189DRGk0h3hefl7Wwlw14oNDwjjHfQknSneGi84pJxGu5WZcRLzjncT1YHygrA+OXptc2T573Lx5k1FKmzf3qICRMvnGhe+YaXPjRyEZTWT6S7d+bbsYdPjAB7+E1uRQgKZyDA+twQh2sy7KpeerXogSt6ajE01IqQDDU3lO2qehdWRMuRVsSj0ooY7whs6ZhlWYRMxN52z/EdXrkOEvqsyNwZ1SNXjZoqsq6m/EaSyY5OMwjHsz08imLPGab8WV+727x810stR1qRDl2hh48wqGBvyU6qA7k8aU6wFleYllsYTAWO7Zkfn+o0UHBdkPc8xqRijWOcOCmLB2qIDaJ5WxiHjip85z1eXfEH+o1/lyWP7ONoj2lICR1lkTFOgS7GYV8CSSHaA76ewwrekZDQ6eymlBYYMsbWHblzA8XqkjSW4p0cKaqbzuG0A86W6B1UsLQa+cbFIN3D+CkqpvmVIvPrxE6yDxK6jm5JpvkP4m+/nXg+gJsTSjWttqiG5YkXEJmDZEgRc8NFDqE50hzA6HoLMDSuuMRYDuM54ZpAK49UbTbZX5SdrI91tNgIGsTWiFPkporWaXdkTCB+aU+sQU2/Sk3Qjej0k5FepxRfYWt4+2L1sGCLdaabRwgSlC2DS3QGO0dPdbACh7c5opOUsD6U3f6rm/ODqADXzcthn5lrYJIBygBDwhFwauwiG7T/m0VHd9K2A7Sg+mKGfyiSps3amuth+t9dRHMQr7ow29Oq9+z3P+oyIHeLtqu350BnWFqiVVlOL0H55Ti4wltHBdbuv5+9ho2hSzK7QjmhOXgfJ1579zIRCqQpQvmVXiJ8jG+g50SfDg2d/ZxiA5xE1BF5PNGM2yS7skPXfHpgEmGIiH3BUjGnXHIt3m01UY4F6yU7W0S6MSqz0NpCJ/e7oD9HB/aDJhbhOmIvzJF3U1hz4gIFwV/YFPQiX2d2W3MN3MZx+iJ3hMtltFKpoIkCKqV2AAYXAQB60xoVJ9+WBP53hw6/IbW+NWeQLpBK20ULCQclDSnsHy5g0pDCIXFFhy+ydOoU54FtEt9Bd9DJ7nfmdu7sURkvdw2OeHogN7Qjk4V/c96x6NixyMI/IHV4eVS/AAIoOOgQI6ujjxZxIdQK223xnXe/EUoFQkDF90L2ERf+qKH7/jwJwr+2loQION5kHR0M2ELqliltROISjnzcPRsqd4oLkaKxZnBBQYCV/nnBJlCMgWbYKU8N3HLAFXlBkqH72cWVK1yFclLqxBqbNbogZHRwcMkbwPLokjcCumCr7cr2sOwiBkV/8P4qVDt718t4ng6zpmaVvEjVYHogzz+gnd0WunI56ThtORnwvAJeCovIu97ZLAIV/FwBkVaS8ZO2Cl68wXPejrP8/wG5EmUtn6gAAA=="


def health():
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/health", timeout=3) as response:
            return json.load(response)
    except Exception:
        return {}


def live_owned(pidfile, expected):
    try:
        pid = int(pidfile.read_text().strip())
        cmdline = Path(f"/proc/{pid}/cmdline").read_bytes().replace(b"\0", b" ").decode(errors="replace")
        return pid if expected in cmdline else None
    except (OSError, ValueError):
        return None


def start_tunnel(work, get_secret):
    token = get_secret("CF_TUNNEL_TOKEN")
    if not token:
        print("⚠️ 服务已启动，但缺少 CF_TUNNEL_TOKEN。请在 Colab Secrets 添加并授权后重跑本格。", flush=True)
        return False
    pidfile = work / "studio-cloudflared.pid"
    if live_owned(pidfile, "cloudflared"):
        print("⏩ 本工作台隧道进程已在运行，连接状态见 studio-cloudflared.log", flush=True)
        return True
    binary = shutil.which("cloudflared")
    if not binary:
        local = Path("/content/cloudflared")
        if not local.is_file() or local.stat().st_size < 1024**2:
            partial = local.with_suffix(".part")
            machine = os.uname().machine
            arch = "arm64" if machine in {"aarch64", "arm64"} else "amd64"
            url = f"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-{arch}"
            subprocess.run(["curl", "-fL", "--retry", "2", "--connect-timeout", "15", "--max-time", "120",
                            "-o", str(partial), url], check=True)
            partial.replace(local)
        local.chmod(0o755)
        binary = str(local)
    tunnel_env = os.environ.copy()
    tunnel_env["TUNNEL_TOKEN"] = token
    log = work / "logs" / "studio-cloudflared.log"
    with log.open("w", encoding="utf-8") as fp:
        # 使用环境变量传入令牌，避免在命令行与日志打印中暴露。
        proc = subprocess.Popen([binary, "tunnel", "--protocol", "http2", "run"],
                                stdout=fp, stderr=subprocess.STDOUT, env=tunnel_env, start_new_session=True)
    pidfile.write_text(str(proc.pid))
    start = time.monotonic()
    while time.monotonic()-start < 30:
        text = log.read_text(encoding="utf-8", errors="replace")
        if proc.poll() is not None:
            print("⚠️ 隧道退出，工作台仍在本地运行。日志：\n" + text[-8000:], flush=True)
            return False
        if "Registered tunnel connection" in text:
            print("✅ Cloudflare HTTP/2 隧道已注册连接", flush=True)
            return True
        time.sleep(.5)
    print(f"⚠️ 隧道连接尚未确认，请查看 {log}；本地工作台已就绪。", flush=True)
    return False


def main():
    if sys.platform != "linux":
        raise RuntimeError("本文件在 Colab A100 Linux 中运行；本地看界面请运行 preview.py")
    payload = json.loads(gzip.decompress(base64.b64decode(BUNDLE_GZ_B64)))
    namespace = {"__name__": "pika_installer", "__file__": "/content/pika-studio-installer.py"}
    exec(compile(payload.pop("installer_runtime.py"), "pika-studio-installer.py", "exec"), namespace)
    config = dict(namespace["CONFIG"], storage_root=STORAGE_ROOT, http_download=HTTP_DOWNLOAD)
    namespace["CONFIG"] = config
    current = health()
    if current.get("service") == "pika-h3-flex-studio":
        work = namespace["select_storage"](config)
        if current.get("version") == APP_VERSION:
            print("⏩ 已有同版本 Pika 工作台在线，无需密码，保留当前任务。", flush=True)
            start_tunnel(work, namespace["secret"])
            print(f"🌐 在电脑上打开工作台：{STUDIO_URL}\n后端隧道：{PUBLIC_URL}", flush=True)
            return
        # Only update our own idle server. Do not interrupt GPU work or queued jobs.
        for path in (work / "jobs").glob("*.json"):
            job = json.loads(path.read_text(encoding="utf-8"))
            if job.get("status") in {"running", "queued", "cancelling"}:
                raise RuntimeError("更新进度条前，请等当前队列完成或在网页取消任务，然后重跑这格；不会中断生成。")
        pid = live_owned(work / "studio-server.pid", str(work / "studio-app" / "server.py"))
        if not pid:
            raise RuntimeError("检测到旧版服务但无法确认归属，请先手动停止旧工作台服务器")
        import signal
        os.kill(pid, signal.SIGTERM)
        for _ in range(40):
            if not health():
                break
            time.sleep(.25)
        else:
            raise RuntimeError("旧版服务尚未停止，请稍后重跑")
        print("✅ 已停止本工作台的空闲旧版服务，正在更新。", flush=True)
    try:
        with socket.create_connection(("127.0.0.1", PORT), timeout=1):
            raise RuntimeError(f"{PORT} 已被其他程序占用，请先停止旧 ComfyUI，再运行工作台；本脚本不会杀其他进程。")
    except OSError:
        pass
    work, env, cli = namespace["prepare_environment"]()
    app_dir = work / "studio-app"
    for name, content in payload.items():
        target = app_dir / name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(content, encoding="utf-8")
    env.pop("PIKA_STUDIO_PASSWORD", None)
    env.pop("CF_TUNNEL_TOKEN", None)
    log = work / "logs" / "studio-server.log"
    with log.open("a", encoding="utf-8") as fp:
        proc = subprocess.Popen([str(cli.parent / "python"), str(app_dir / "server.py"),
                                 "--root", str(work), "--port", str(PORT)],
                                stdout=fp, stderr=subprocess.STDOUT, env=env, start_new_session=True)
    (work / "studio-server.pid").write_text(str(proc.pid))
    for _ in range(120):
        if proc.poll() is not None:
            print(log.read_text(encoding="utf-8", errors="replace")[-10000:], flush=True)
            raise RuntimeError("工作台服务提前退出")
        if health().get("service") == "pika-h3-flex-studio":
            break
        time.sleep(.5)
    else:
        print(log.read_text(encoding="utf-8", errors="replace")[-10000:], flush=True)
        raise RuntimeError("工作台启动超时")
    print("✅ 工作台已启动，生成任务在网页提交，不会自动跑示例。", flush=True)
    print("🌐 工作台无需登录，打开网页即可使用。", flush=True)
    try:
        connected = start_tunnel(work, namespace["secret"])
    except Exception as exc:
        connected = False
        print(f"⚠️ 隧道准备失败：{exc}。本地工作台仍在运行。", flush=True)
    print(f"🌐 在电脑上打开工作台：{STUDIO_URL}\n后端隧道：{PUBLIC_URL}", flush=True)
    print(f"隧道 {'已连接' if connected else '尚待确认'}；Cloudflare 服务地址应为 http://127.0.0.1:{PORT}", flush=True)
    print(f"模型缓存复用：{work / 'hf'}\n服务日志：{log}\n生成视频：{work / 'outputs'}", flush=True)


if __name__ == "__main__":
    main()
